# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Official install (`uv` + Python 3.10 venv + `stable-worldmodel[train,format]` + per-benchmark env deps) with the compatibility pins found on Colab (`transformers<5` etc.), the dataset-path fix, and Colab-safe training knobs. Runtime → GPU; start with `BENCH='tworoom'` + `EPOCHS=5`.

GPU: the model is ~15M params, so a **T4 (free)** has ample memory; **L4/A100** are faster. Validation (`EPOCHS=5`, 2 variants) ≈ 30–75 min on T4 (data loading + planning eval dominate); matched-compute (`EPOCHS=100`) ≈ a few hours per variant.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle — REAL install (uv/3.10 venv) + robust assets + train/eval
import os, subprocess, sys, glob, shutil

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G) | pusht (13G) | reacher (24G) | cube (46G)
EPOCHS   = 5           # 5 = validate; 100 = matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]
GET_DATA = True
BRANCH   = "claude/upbeat-babbage-kbmgsr"
GH_TOKEN = ""
BATCH_SIZE = 32        # Colab-safe; official is 128
PRECISION  = "32"      # after one good run, try "bf16-mixed" on L4/A100/H100
# ------------------------------------------------------------------
DATACFG  = {"tworoom":"tworoom","pusht":"pusht","reacher":"dmc","cube":"ogb"}[BENCH]
ENV_DEPS = {"tworoom":"pygame pymunk shapely","pusht":"pygame pymunk shapely",
            "reacher":"dm_control mujoco","cube":"ogbench"}[BENCH]
H="/content/stable-wm"; REPO="/content/spinangle"; VENV="/content/lewmenv"; PY=f"{VENV}/bin/python"
os.environ.update(STABLEWM_HOME=H, MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl", MPLBACKEND="Agg",
                  HYDRA_FULL_ERROR="1", WANDB_MODE="disabled",
                  HF_HUB_DOWNLOAD_TIMEOUT="60", HF_HUB_ETAG_TIMEOUT="60")
def run(c, check=True):
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    return subprocess.run(c, shell=True, check=check, env=os.environ).returncode
def cap(c, tail=20000):
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    r=subprocess.run(c, shell=True, capture_output=True, text=True, env=os.environ)
    print(((r.stdout or "")+(r.stderr or ""))[-tail:]); print("exit", r.returncode); return r.returncode
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
    HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    if HF_TOKEN: os.environ["HF_TOKEN"]=HF_TOKEN; os.environ["HUGGINGFACE_HUB_TOKEN"]=HF_TOKEN
except Exception: pass

run("nvidia-smi -L || echo '⚠️ NO GPU — Runtime > Change runtime type > GPU'", check=False)
if not os.path.isdir(f"{REPO}/.git"):
    auth=f"{GH_TOKEN}@" if GH_TOKEN else ""
    run(f"git clone -b {BRANCH} https://{auth}github.com/turtlenottortoise/spinangle.git {REPO}")
else:
    run(f"cd {REPO} && git fetch origin && git checkout {BRANCH} && git pull --ff-only", check=False)
os.chdir(REPO)
run("apt-get -qq update && apt-get -qq install -y xvfb zstd swig ffmpeg patchelf "
    "libegl1 libgl1-mesa-glx libosmesa6 libglfw3 libglew2.2 >/dev/null 2>&1", check=False)
if not os.path.exists(PY):
    run("pip install -q uv"); run("uv python install 3.10"); run(f"uv venv --python 3.10 {VENV}")

# install: [train]+[format] (required) then env deps (full [env] -> scoped fallback)
if cap(f"uv pip install --python {PY} 'stable-worldmodel[train,format]' matplotlib huggingface_hub imageio-ffmpeg"):
    if cap(f"uv pip install --python {PY} 'stable-worldmodel[train]' matplotlib huggingface_hub imageio imageio-ffmpeg h5py"):
        raise SystemExit("❌ core install failed — paste output above")
if cap(f"uv pip install --python {PY} 'stable-worldmodel[env]'"):
    print(f"\n[note] full [env] failed (Crafter/Atari/Box2D, not LeWM). Installing: {ENV_DEPS}")
    if cap(f"uv pip install --python {PY} {ENV_DEPS}"):
        raise SystemExit(f"❌ env deps for {BENCH} failed — paste output above")

# Compatibility pins (found while bringing the stack up on Colab):
#  transformers<5 keeps the OLD ViT state-dict keys so the published checkpoint loads;
#  datasets>=2.20 exposes datasets.config; pyarrow==20 keeps PyExtensionType;
#  huggingface_hub<1 plays well with transformers 4.x.
run(f"uv pip install --python {PY} --upgrade --reinstall 'transformers<5.0.0' "
    f"'datasets>=2.20.0,<3.0.0' 'pyarrow==20.0.0' 'huggingface_hub>=0.34.0,<1.0.0' hf_xet")

IMPORT_CHECK=(f"{PY} - <<'EOF'\n"
 "import importlib,sys,traceback\nbad=[]\n"
 "for m in ['torch','torchvision','hydra','omegaconf','transformers','lightning','imageio','h5py','datasets','pyarrow','stable_pretraining','stable_worldmodel']:\n"
 "    try: x=importlib.import_module(m); print('OK  ',m,getattr(x,'__version__',''))\n"
 "    except Exception: bad.append(m); print('FAIL',m); traceback.print_exc(file=sys.stdout)\n"
 "import torch; print('CUDA', torch.cuda.is_available())\n"
 "import pyarrow as pa; assert hasattr(pa,'PyExtensionType'); print('pyarrow.PyExtensionType OK')\n"
 "import jepa, module; print('repo imports OK')\nsys.exit(1 if bad else 0)\nEOF")
if cap(IMPORT_CHECK):
    raise SystemExit("❌ imports/compat failed — paste the traceback above")

run(f"{PY} smoke_test.py && {PY} metrics.py")

# data + checkpoint (download_assets prints stage + traceback on failure)
if cap(f"{PY} -u scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else "")):
    raise SystemExit("❌ download_assets failed — paste output above")

# stable_worldmodel loads datasets from $STABLEWM_HOME/datasets/<name>.h5 — make sure
# the extracted file is reachable there (symlink; copy if symlink unsupported).
if GET_DATA:
    os.makedirs(f"{H}/datasets", exist_ok=True)
    src = next((p for p in [f"{H}/{DATACFG}.h5", f"{H}/{BENCH}.h5", f"{H}/datasets/{DATACFG}.h5"]
                if os.path.exists(p) and os.path.getsize(p) > 1<<20), None)
    dst = f"{H}/datasets/{DATACFG}.h5"
    if src and os.path.abspath(src) != os.path.abspath(dst):
        if os.path.lexists(dst): os.remove(dst)
        try: os.symlink(src, dst); print(f"symlinked {dst} -> {src}")
        except Exception: shutil.copy2(src, dst); print(f"copied {dst}")
    elif not src:
        raise SystemExit(f"❌ dataset .h5 not found under {H} after extract")

# PHASE 1 reproduce (planning eval is non-fatal so latent metrics + plots still run)
cap(f"xvfb-run -a {PY} -u eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# Colab-safe training knobs (single-process loader avoids Colab multiprocessing issues)
COMMON = (f"data={DATACFG} trainer.max_epochs={EPOCHS} wandb.enabled=false "
          f"loader.batch_size={BATCH_SIZE} loader.num_workers=0 loader.persistent_workers=false "
          f"loader.prefetch_factor=null trainer.precision={PRECISION} trainer.devices=1")
CK=f"{H}/checkpoints/{BENCH}"
for v in VARIANTS:
    print(f"\n========== TRAIN {v} ==========")
    if cap(f"{PY} -u train.py +experiment={v} {COMMON} output_model_name={BENCH}/{v}"):
        raise SystemExit(f"❌ training failed for {v} — traceback above")
    for old in sorted(glob.glob(f"{CK}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)[:-1]: os.remove(old)
    sph="" if v in ("official_lewm","lewm_nosigreg") else "--spherical"
    cap(f"{PY} -u scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
        f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16")
    cap(f"xvfb-run -a {PY} -u eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}")

cap(f"{PY} -u scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps","rollout_error_vs_horizon","retrieval_vs_steps","rank_clumping","planning_budget_curve"]:
    fp=f"{REPO}/plots/{p}.png"
    if os.path.exists(fp): display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/.")


## Phase 7 — νGPT scaling (optional; run after the loop above)

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
PY, BENCH, DATACFG, EPOCHS = '/content/lewmenv/bin/python', 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !{PY} train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} loader.num_workers=0 wandb.enabled=false
    !xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
